# 03 - Model Training & Evaluation

Train and evaluate multiple ML models for chemical property prediction.

**Models:**
- Random Forest
- Gradient Boosting
- XGBoost

**Features:**
- Cross-validation
- Hyperparameter tuning
- Model comparison
- SHAP explainability

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.preprocessor import DataPreprocessor
from src.models.trainer import ModelTrainer, MultiTargetTrainer
from src.models.explainability import SHAPExplainer
from src.visualization.plots import Visualizer
from src.utils.config import settings

print("Setup complete")

## 1. Load Processed Features

In [ ]:
# Load processed data
feature_path = settings.PROCESSED_DATA_DIR / 'features_selected.csv'

if feature_path.exists():
    df = pd.read_csv(feature_path)
    print(f"Loaded {df.shape}")
else:
    print("Using sample data...")
    np.random.seed(42)
    df = pd.DataFrame(
        np.random.randn(100, 20),
        columns=[f'feature_{i}' for i in range(20)]
    )
    df['target'] = np.random.randn(100)

# Identify target column
target_candidates = ['solubility', 'boiling_point', 'toxicity_category', 'target']
target_col = next((c for c in target_candidates if c in df.columns), df.columns[-1])
feature_cols = [c for c in df.columns if c != target_col]

print(f"Target: {target_col}")
print(f"Features: {len(feature_cols)}")
df.head()

## 2. Preprocess Data

In [ ]:
# Prepare data
preprocessor = DataPreprocessor(
    scaler_type='standard',
    test_size=0.2,
    val_size=0.2,
    random_state=42,
)

X = df[feature_cols]
y = df[target_col]

# Determine problem type
problem_type = 'classification' if y.nunique() <= 10 else 'regression'
print(f"Problem type: {problem_type}")
print(f"Target unique values: {y.nunique()}")

# Split data
splits = preprocessor.split_data(X, y)

# Scale features
scaled = preprocessor.scale_features(
    splits['X_train'], splits['X_val'], splits['X_test']
)

print(f"Train: {len(scaled['X_train'])}, Val: {len(scaled.get('X_val', []))}, Test: {len(scaled['X_test'])}")

## 3. Train Random Forest

In [ ]:
rf_trainer = ModelTrainer(
    model_type='random_forest',
    problem_type=problem_type,
    random_state=42,
)

rf_results = rf_trainer.train(
    scaled['X_train'], splits['y_train'],
    scaled.get('X_val'), splits.get('y_val'),
    tune_hyperparams=False,
)

print("Random Forest Results:")
for metric, value in rf_results['metrics'].items():
    print(f"  {metric}: {value:.4f}")

## 4. Train Gradient Boosting

In [ ]:
gb_trainer = ModelTrainer(
    model_type='gradient_boosting',
    problem_type=problem_type,
    random_state=42,
)

gb_results = gb_trainer.train(
    scaled['X_train'], splits['y_train'],
    scaled.get('X_val'), splits.get('y_val'),
    tune_hyperparams=False,
)

print("Gradient Boosting Results:")
for metric, value in gb_results['metrics'].items():
    print(f"  {metric}: {value:.4f}")

## 5. Train XGBoost

In [ ]:
xgb_trainer = ModelTrainer(
    model_type='xgboost',
    problem_type=problem_type,
    random_state=42,
)

xgb_results = xgb_trainer.train(
    scaled['X_train'], splits['y_train'],
    scaled.get('X_val'), splits.get('y_val'),
    tune_hyperparams=False,
)

print("XGBoost Results:")
for metric, value in xgb_results['metrics'].items():
    print(f"  {metric}: {value:.4f}")

## 6. Compare Models

In [ ]:
# Compare results
all_results = {
    'Random Forest': rf_results['metrics'],
    'Gradient Boosting': gb_results['metrics'],
    'XGBoost': xgb_results['metrics'],
}

comparison_df = pd.DataFrame(all_results).T
print("Model Comparison:")
display(comparison_df)

# Visualization
viz = Visualizer()
fig = viz.plot_model_comparison(all_results)
plt.show()

## 7. Feature Importance

In [ ]:
# Get feature importance from best model
importance_df = xgb_trainer.get_feature_importance(feature_cols)

fig = viz.plot_feature_importance(importance_df.head(20))
plt.show()

## 8. SHAP Explainability

In [ ]:
# SHAP explanation for XGBoost
explainer = SHAPExplainer(
    xgb_trainer.model,
    feature_names=feature_cols,
)

# Summary plot
fig = explainer.explain_global(
    scaled['X_test'],
    max_display=15,
)
plt.show()

## 9. Local Explanation

In [ ]:
# Explain first test instance
fig = explainer.explain_local(
    scaled['X_test'].values[:1],
    feature_names=feature_cols,
)
plt.show()

## 10. Save Models

In [ ]:
# Save all models
for name, trainer in [('rf', rf_trainer), ('gb', gb_trainer), ('xgb', xgb_trainer)]:
    path = settings.MODEL_DIR / f'{target_col}_{name}.joblib'
    trainer.save_model(path)
    print(f"Saved {name} to {path}")

print("\nTraining complete!")